In [ ]:
### Geolibraries
import geopandas as gpd
import osmnx as ox
import contextily as ctx


# General tools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pyarrow.parquet as pq

from h3 import h3
from shapely.geometry import Polygon

In [ ]:
users = pd.read_parquet("./data/user_pois_pt_1_tampere.parquet")

In [ ]:
home_users_df = (
    users[users["is_home"] == 1]
    .drop_duplicates(subset="user_id")
)

In [ ]:
home_users_df

In [ ]:
#bring the POI layer
pois = pd.read_parquet("data/pois_per_hex_new_class_tampere.parquet")

In [ ]:
pois_wide = (
    pois
    .pivot_table(
        index="h3_id",
        columns="category",
        values="count",
        fill_value=0
    )
    .reset_index()
)

In [ ]:
pois_wide

In [ ]:

cols_needed = [
    "from_id","to_id", "co2_emissions_g"
]

table = pq.read_table("./output/bike_co2_3000_tampere.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_bike_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
# create symmetric pairs (outbound + inbound)
df_bike_co2_sym = df_bike_co2.merge(
    df_bike_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)

In [ ]:
df_bike_co2_sym["bike_co2_outbound"] = df_bike_co2_sym["co2_emissions_g_outbound"]

df_bike_co2_sym["bike_co2_inbound"] = (
    df_bike_co2_sym["co2_emissions_g_inbound"]
    .fillna(df_bike_co2_sym["co2_emissions_g_outbound"])
)

df_bike_co2_sym["bike_co2_total"] = (
    df_bike_co2_sym["bike_co2_outbound"]
    + df_bike_co2_sym["bike_co2_inbound"]
)


In [ ]:
df_bike_co2_final = (
    df_bike_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "bike_co2_outbound",
            "bike_co2_inbound",
            "bike_co2_total"
        ]
    ]
)


In [ ]:
df_bike_co2 = df_bike_co2_final.copy()

In [ ]:
# value to assign for bike (could be very low per km)
bike_co2_value = 0.3 * 26  # adjust as needed

# 1. update existing rows where from_id == to_id
mask_same = df_bike_co2["from_id"] == df_bike_co2["to_id"]
df_bike_co2.loc[mask_same, "bike_co2_total"] = bike_co2_value

# 2. find which from_ids still need new rows (those not in existing from_id==to_id)
from_ids = df_bike_co2["from_id"].unique()
existing_pairs = set(zip(df_bike_co2["from_id"], df_bike_co2["to_id"]))

new_rows = [
    {"from_id": fid, "to_id": fid, "bike_co2_total": bike_co2_value}
    for fid in from_ids
    if (fid, fid) not in existing_pairs
]

# 3. create and append missing rows
df_synthetic_bike = pd.DataFrame(new_rows)
df_bike_co2 = pd.concat([df_bike_co2, df_synthetic_bike], ignore_index=True)


In [ ]:
df_bike_co2

In [ ]:
# filter rows where from_id == to_id
same_id_rows = df_bike_co2[df_bike_co2["from_id"] == df_bike_co2["to_id"]]

# show them
same_id_rows

In [ ]:
poi_long = (
    pois_wide
    .melt(id_vars='h3_id', var_name='category', value_name='n_pois')
    .query('n_pois > 0')
    [['h3_id', 'category']]
)


In [ ]:
poi_long

In [ ]:
od_home_bike = df_bike_co2[
    df_bike_co2['from_id'].isin(home_users_df['home_gid9'])
]


In [ ]:
od_poi_bike = od_home_bike.merge(
    poi_long,
    left_on='to_id',
    right_on='h3_id',
    how='inner'
)


In [ ]:
od_poi_bike

In [ ]:
nearest_bike = (
    od_poi_bike
    .sort_values('bike_co2_total')
    .groupby(['from_id', 'category'], as_index=False)
    .first()
)


In [ ]:
nearest_wide_bike = (
    nearest_bike
    .pivot(index='from_id', columns='category', values='bike_co2_total')
    .reset_index()
)

In [ ]:
nearest_wide_bike

In [ ]:
final_df_bike = home_users_df.merge(
    nearest_wide_bike,
    left_on='home_gid9',
    right_on='from_id',
    how='left'
)

In [ ]:
final_df_bike

In [ ]:
final_df_bike = final_df_bike.rename(columns={
    'Education_y': 'Education_nearest',
    'Healthcare and Health_y': 'Healthcare and Health_nearest',
    'Others / Not sure_y': 'Others / Not sure_nearest',
    'Recreational, Outdoors_y': 'Recreational, Outdoors_nearest',
    'Shopping, Errands_y': 'Shopping, Errands_nearest',
    'Social, Cultural_y': 'Social, Cultural_nearest'
})

In [ ]:
### Jobs

In [ ]:

cols_needed = [
    "from_id","to_id", "co2_emissions_g"
]

table = pq.read_table("./output/bike_co2_3000_tampere.parquet", 
                      columns=cols_needed,
                      use_threads=True)

df_bike_co2 = table.to_pandas(types_mapper=pd.ArrowDtype)  # keeps pandas light

In [ ]:
df_bike_co2_sym = df_bike_co2.merge(
    df_bike_co2,
    left_on=["from_id", "to_id"],
    right_on=["to_id", "from_id"],
    how="left",
    suffixes=("_outbound", "_inbound")
)


In [ ]:
df_bike_co2_sym

In [ ]:
df_bike_co2_sym["bike_co2_outbound"] = df_bike_co2_sym["co2_emissions_g_outbound"]

df_bike_co2_sym["bike_co2_inbound"] = (
    df_bike_co2_sym["co2_emissions_g_inbound"]
    .fillna(df_bike_co2_sym["co2_emissions_g_outbound"])
)

df_bike_co2_sym["bike_co2_total"] = (
    df_bike_co2_sym["co2_emissions_g_outbound"]
    + df_bike_co2_sym["co2_emissions_g_inbound"]
)

In [ ]:
df_bike_co2_final = (
    df_bike_co2_sym
    .rename(columns={
        "from_id_outbound": "from_id",
        "to_id_outbound": "to_id"
    })[
        [
            "from_id",
            "to_id",
            "bike_co2_outbound",
            "bike_co2_inbound",
            "bike_co2_total"
        ]
    ]
)


In [ ]:
df_bike_co2 = df_bike_co2_final.copy()

In [ ]:
df_jobs = pd.read_parquet("./data/users_and_works_tampere.parquet")

In [ ]:
df_jobs_co2_bike = df_jobs.merge(
    df_bike_co2,
    left_on=["home_gid9", "work_gid9"],
    right_on=["from_id", "to_id"],
    how="left"
)


In [ ]:
df_jobs_co2_bike.columns

In [ ]:
jobs_co2_bike = df_jobs_co2_bike[['user_id', 'bike_co2_total']].rename(
    columns={'bike_co2_total': 'job_real_nearest_bike'}
)

In [ ]:
final_df_bike = final_df_bike.merge(
    jobs_co2_bike,
    on='user_id',
    how='left'
)


In [ ]:
nearest_cols_bike = [c for c in final_df_bike.columns if c.endswith('_nearest_bike')]

final_df_bike = final_df_bike.dropna(subset=nearest_cols_bike)


In [ ]:
worker_profile = {
    "jobs": 4,
    "Social, Cultural": 2,
    "Shopping, Errands": 1,
    "Recreational, Outdoors": 2
}

In [ ]:
final_df_bike

In [ ]:
final_df_bike.to_parquet("./output/bike_expenditure_weekly_nearest_tampere.parquet")

In [ ]:
final_df_bike['weekly_co2_expenditure_bike'] = (
    worker_profile["jobs"] * final_df_bike['job_real_nearest_bike'] +
    worker_profile["Social, Cultural"] * final_df_bike['Social, Cultural_nearest'] +
    worker_profile["Shopping, Errands"] * final_df_bike['Shopping, Errands_nearest'] +
    worker_profile["Recreational, Outdoors"] * final_df_bike['Recreational, Outdoors_nearest']
)


In [ ]:
final_df_bike.to_parquet("./output/bike_expenditure_weekly_nearest_tampere.parquet")

In [ ]:
import matplotlib.pyplot as plt

# --- Data ---
df = final_df_bike.copy()
df = df[df["weekly_co2_expenditure_bike"].notna()]

# Optional: remove extreme outliers for readability
df = df[
    df["weekly_co2_expenditure_bike"]
    <= df["weekly_co2_expenditure_bike"].quantile(0.99)
]

values = df["weekly_co2_expenditure_bike"]

# --- Figure ---
fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    values,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# --- Climate budgets ---
ax.axvline(
    7000,
    linestyle="--",
    linewidth=2,
    label="2030 budget (7 kg CO₂ / week)"
)

ax.axvline(
    3000,
    linestyle="--",
    linewidth=2,
    label="2050 budget (3 kg CO₂ / week)"
)

# --- Formatting ---
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ expenditure per worker (grams)", fontsize=11)
ax.set_title(
    "Minimum Weekly CO₂ Needed to Meet Essential Activities (Bike)",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

# --- Budgets (grams CO₂ / week) ---
BUDGET_2030 = 7000
BUDGET_2050 = 3000

# --- Data ---
df = final_df_bike.copy()
df = df[df["weekly_co2_expenditure_bike"].notna()]

# optional: trim extreme outliers for readability
df = df[
    df["weekly_co2_expenditure_bike"]
    <= df["weekly_co2_expenditure_bike"].quantile(0.99)
]

values = df["weekly_co2_expenditure_bike"]
total_workers = len(values)

# --- Percentages & counts ---
pct_below_2030 = (values <= BUDGET_2030).mean() * 100
pct_below_2050 = (values <= BUDGET_2050).mean() * 100

count_below_2030 = (values <= BUDGET_2030).sum()
count_above_2030 = (values > BUDGET_2030).sum()

count_below_2050 = (values <= BUDGET_2050).sum()
count_above_2050 = (values > BUDGET_2050).sum()

# --- Figure ---
fig, ax = plt.subplots(figsize=(10, 4))

# Boxplot
ax.boxplot(
    values,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", edgecolor="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black"),
    flierprops=dict(marker=".", markersize=3, alpha=0.4)
)

# --- Budget lines with colors ---
ax.axvline(
    BUDGET_2030,
    color="tab:orange",
    linestyle="--",
    linewidth=2,
    label="2030 budget"
)

ax.axvline(
    BUDGET_2050,
    color="tab:blue",
    linestyle="--",
    linewidth=2,
    label="2050 budget"
)

# --- Annotations under plot ---
ax.text(
    0.68, -0.12,
    f"{pct_below_2030:.1f}% below 2030 ({count_below_2030:,})\n"
    f"{count_above_2030:,} above",
    color="tab:orange",
    fontsize=10,
    transform=ax.transAxes,
    va="top"
)

ax.text(
    0.68, -0.24,
    f"{pct_below_2050:.1f}% below 2050 ({count_below_2050:,})\n"
    f"{count_above_2050:,} above",
    color="tab:blue",
    fontsize=10,
    transform=ax.transAxes,
    va="top"
)

# --- Formatting ---
ax.set_yticks([])
ax.set_xlabel("Weekly CO₂ expenditure per worker (grams)", fontsize=11)
ax.set_title(
    "Minimum Weekly CO₂ Needed to Meet Essential Activities (Bike)",
    fontsize=14
)

ax.legend(frameon=False)
ax.grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()



In [ ]:
### Ensure the same user base 